**Installations**

In [ ]:
pip install rouge_score bert_score sentence_transformers

In [ ]:
%pip uninstall -y openai httpx httpcore
%pip install --upgrade openai==1.52.0 httpx==0.27.2 httpcore==1.0.5


In [ ]:
# ============================================
# STEP 1: INSTALL ALL PACKAGES
# ============================================
# Run this cell first, then RESTART YOUR KERNEL before proceeding to Step 2

# Uninstall any conflicting packages first
!pip uninstall -y numpy thinc spacy scispacy

# Install numpy first (this is critical for compatibility)
!pip install numpy==1.26.4

# Install newer compatible versions
!pip install spacy==3.7.2
!pip install scispacy==0.5.4

# Install the corresponding biomedical model for scispacy 0.5.4
!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz

# Install transformers and related packages
!pip install transformers datasets torch pandas scikit-learn rouge-score nltk

# Install additional ML packages
!pip install sentence-transformers
!pip install accelerate bitsandbytes bert-score peft

print("\n Installation complete! PLEASE RESTART YOUR KERNEL NOW before running the next cell.")

**Importing Libraries**

In [ ]:
# ============================================
# STEP 2: IMPORT LIBRARIES AND DOWNLOAD NLTK DATA
# ============================================

import os
import re
import logging
import numpy as np
import pandas as pd
import torch
from torch import cuda
from torch.utils.data import DataLoader, Dataset

# NLP / evaluation libraries
import nltk
from nltk.tokenize import word_tokenize
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from bert_score import score as bert_score
from sentence_transformers import SentenceTransformer, util
import spacy

# Model / PEFT / quantization
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# OpenAI (for LLM judge or GEval)
from openai import OpenAI

# Suppress transformers warnings
logging.getLogger("transformers").setLevel(logging.ERROR)

# Download required NLTK data
nltk.download("punkt")
nltk.download("wordnet")

print("✅ All imports successful! You're ready to proceed.")


In [ ]:
# ============================================
# STEP 3: VERIFY INSTALLATIONS
# ============================================
# Optional: Run this to verify everything is working

import numpy as np

print("Checking versions:")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Spacy version: {spacy.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {__import__('transformers').__version__}")

# Test spacy import
try:
    nlp = spacy.load("en_ner_bc5cdr_md")
    print("Scispacy model loaded successfully!")
except Exception as e:
    print(f"Error loading scispacy model: {e}")

In [ ]:
# Load spaCy and SentenceTransformer models
def load_spacy_model():
    """Load spaCy medical NER model."""
    try:
        return spacy.load("en_ner_bc5cdr_md")
    except Exception as e:
        print(f"Error loading spaCy model: {e}")
        return None

def load_sentence_transformer():
    """Load SentenceTransformer for FCS."""
    try:
        return SentenceTransformer("all-MiniLM-L6-v2", device="cuda" if torch.cuda.is_available() else "cpu")
    except Exception as e:
        print(f"Error loading SentenceTransformer: {e}")
        return None

nlp = load_spacy_model()
embedder = load_sentence_transformer()

In [ ]:
# ============================================
# LOAD FINE-TUNED MODEL FROM HUGGING FACE
# ============================================
# Replace "your-username/your-model-name" with your actual Hugging Face model identifier
# Examples:
#   - For a PEFT adapter: "your-username/medllama-7b-finetuned-adapter"
#   - For a full model: "your-username/medllama-7b-finetuned"

from huggingface_hub import login
import os

# Set your Hugging Face model identifier here
HF_FINETUNED_MODEL = "AparnaSuresh/MedLlama-3b"
# Set your Hugging Face token from environment variable (secure)
# Get it from https://huggingface.co/settings/tokens and set as environment variable
HF_TOKEN = os.getenv("HF_TOKEN", "your_hf_token_here")  # Get from environment variable

# Authenticate with Hugging Face
if HF_TOKEN != "your_hf_token_here":
    login(token=HF_TOKEN)
    print("✓ Authenticated with Hugging Face")
else:
    print("⚠ Warning: HF_TOKEN not set. Using anonymous access (may fail for private models).")
    print("   To authenticate, replace 'your_hf_token_here' with your actual token.")

print(f"Will load fine-tuned model from: {HF_FINETUNED_MODEL}")


In [ ]:
# ============================================
# FIX: Update bitsandbytes
# ============================================
# Run this cell and RESTART YOUR KERNEL after

!pip install -U bitsandbytes

print("\nbitsandbytes updated! PLEASE RESTART YOUR KERNEL before continuing.")

In [ ]:
!pip install protobuf

# Load fine-tuned model and tokenizer from Hugging Face
def load_fine_tuned_model(base_model_name="meta-llama/Meta-Llama-3-8B", finetuned_model_name=None, is_merged_model=True):
    """
    Load the fine-tuned model and tokenizer from Hugging Face.

    Args:
        base_model_name: Base model identifier on Hugging Face (only used if is_merged_model=False)
        finetuned_model_name: Fine-tuned model identifier on Hugging Face
        is_merged_model: If True (default), loads as full merged model. If False, tries PEFT adapter.
    """
    try:
        if finetuned_model_name is None:
            finetuned_model_name = HF_FINETUNED_MODEL

        print(f"Loading fine-tuned model from Hugging Face: {finetuned_model_name}")

        if is_merged_model:
            # Load merged model directly (adapter already merged into base)
            print("Loading merged model (full fine-tuned model)...")
            tokenizer = AutoTokenizer.from_pretrained(finetuned_model_name, trust_remote_code=True)
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token

            print("Configuring quantization...")
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=False,
            )

            # Load the merged model
            model = AutoModelForCausalLM.from_pretrained(
                finetuned_model_name,
                quantization_config=bnb_config,
                device_map="auto",
                torch_dtype=torch.float16,
                trust_remote_code=True,
                low_cpu_mem_usage=True,
            )
            model.config.pad_token_id = tokenizer.eos_token_id
            print("✓ Loaded merged model successfully!")
            return model, tokenizer
        else:
            # Try to load as full model first, then fallback to PEFT adapter
            try:
                print("Attempting to load as full fine-tuned model...")
                tokenizer = AutoTokenizer.from_pretrained(finetuned_model_name, trust_remote_code=True)
                if tokenizer.pad_token is None:
                    tokenizer.pad_token = tokenizer.eos_token

                print("Configuring quantization...")
                bnb_config = BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_quant_type="nf4",
                    bnb_4bit_compute_dtype=torch.float16,
                    bnb_4bit_use_double_quant=False,
                )

                model = AutoModelForCausalLM.from_pretrained(
                    finetuned_model_name,
                    quantization_config=bnb_config,
                    device_map="auto",
                    torch_dtype=torch.float16,
                    trust_remote_code=True,
                    low_cpu_mem_usage=True,
                )
                model.config.pad_token_id = tokenizer.eos_token_id
                print("✓ Loaded as full fine-tuned model!")
                return model, tokenizer

            except Exception as full_model_error:
                print(f"Full model load failed (trying PEFT adapter): {full_model_error}")
                print("Attempting to load as PEFT adapter...")

                # Load base model first
                print(f"Loading base model: {base_model_name}")
                tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
                if tokenizer.pad_token is None:
                    tokenizer.pad_token = tokenizer.eos_token

                print("Configuring quantization...")
                bnb_config = BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_quant_type="nf4",
                    bnb_4bit_compute_dtype=torch.float16,
                    bnb_4bit_use_double_quant=False,
                )

                print("Loading base model...")
                base_model = AutoModelForCausalLM.from_pretrained(
                    base_model_name,
                    quantization_config=bnb_config,
                    device_map="auto",
                    torch_dtype=torch.float16,
                    trust_remote_code=True,
                    low_cpu_mem_usage=True,
                )
                base_model.config.pad_token_id = tokenizer.eos_token_id

                # Load PEFT adapter from Hugging Face
                print(f"Loading PEFT adapter from: {finetuned_model_name}")
                model = PeftModel.from_pretrained(base_model, finetuned_model_name, is_trainable=False)
                print("✓ Loaded as base model + PEFT adapter!")
                return model, tokenizer

    except Exception as e:
        print(f"❌ Error loading fine-tuned model: {e}")
        import traceback
        traceback.print_exc()
        return None, None

# Load the merged model from Hugging Face (adapter already merged)
model, tokenizer = load_fine_tuned_model(
    base_model_name="meta-llama/Meta-Llama-3-8B",
    finetuned_model_name=HF_FINETUNED_MODEL,
    is_merged_model=True  # Set to True if you have a merged model, False if it's just a PEFT adapter
)
if model is None or tokenizer is None:
    raise ValueError("Failed to load fine-tuned model or tokenizer. Please check your Hugging Face model identifier.")

print("\n✓ Ready for inference!")

In [ ]:
questions = [
    "I have a severe headache. What should I do?",
    "My 4-year-old has a fever of 102°F for 2 days. What should I do?",
    "I’ve had diarrhea since yesterday. What can I take?",
    "I cut my finger and it’s bleeding. What should I do?"
]

references = [
    # 1) Severe headache
    "Rest in a dark, quiet room, hydrate with water or an electrolyte drink, and consider acetaminophen 500–1,000 mg every 6–8 hours (max 3,000 mg/day) or ibuprofen 200–400 mg every 6–8 hours with food (max 1,200 mg/day OTC; avoid in pregnancy, ulcers, kidney disease, or if on blood thinners). Seek urgent care if this is a sudden “worst headache,” if there are new neurologic symptoms, fever with neck stiffness, head injury, or pregnancy with high blood pressure. Limit screens, track triggers like stress or caffeine, and arrange follow-up if headaches are frequent or disabling.",
    # 2) Child fever
    "Offer frequent fluids, light clothing, and lukewarm sponging for comfort. For fever control, use acetaminophen 10–15 mg/kg every 4–6 hours (max 5 doses/day) or ibuprofen 10 mg/kg every 6–8 hours if age ≥6 months, avoiding aspirin. Seek urgent care for temperature ≥104°F (40°C), lethargy, neck stiffness, breathing difficulty, dehydration signs, seizure, or a rash that does not blanch. If the child is drinking and otherwise well, home monitoring is reasonable, but arrange pediatric review if symptoms persist beyond 48–72 hours or worsen.",
    # 3) Diarrhea
    "Start with oral rehydration solution in small, frequent sips and introduce bland foods as tolerated while avoiding alcohol, very fatty foods, and large amounts of sugar. If there is no blood or high fever, loperamide 4 mg once then 2 mg after each loose stool (max 8 mg/day OTC) can reduce frequency; avoid it with bloody diarrhea or suspected invasive infection. Seek care for dehydration, severe or persistent pain, fever ≥101.5°F (38.6°C), symptoms lasting more than 48–72 hours, recent antibiotic use, or concerning travel exposures. Hand hygiene helps prevent spread.",
    # 4) Finger cut
    "Apply firm direct pressure with a clean cloth or gauze for ten minutes without checking, then rinse under running water once bleeding slows, remove visible debris, apply a thin layer of antiseptic or antibiotic ointment, and cover with a clean dressing. Go for care if the wound is deep or gaping, bleeding continues despite pressure, there is numbness or weakness, or contamination from bites or dirty metal. Update tetanus if it has been more than ten years, or more than five years for dirty wounds. Keep the area clean and dry and change the dressing daily while watching for infection."
]


In [ ]:
# ============================================
# OPTION 1: Try Alpaca Format for Fine-Tuned Model
# ============================================
# If your model was trained with Alpaca format, use this instead

def create_prompt_simple(user_query: str) -> str:
    """Simple prompt format (current fine-tuned format)."""
    return (
        "You are a careful, concise triage assistant for the public.\n"
        "Write ONE short paragraph (3–6 sentences). Be direct, practical, and safe.\n"
        "Include: immediate self-care, specific OTC dosing if appropriate, and clear red flags for urgent care.\n"
        "Rules: Do NOT ask the user any questions. Do NOT use bullet points, tables, or meta text.\n\n"
        f"User: {user_query}\n"
        "Answer (one short paragraph, no questions): "
    )

def create_prompt_alpaca_style(user_query: str) -> str:
    """Alpaca-style prompt (same as base model)."""
    instr = (
        "You are a careful, concise triage assistant for the public. "
        "Write ONE short paragraph of 3–6 sentences. Be practical and safe. "
        "Include immediate self-care, specific OTC dosing when appropriate, and clear red flags that require urgent care. "
        "Do NOT ask questions. Do NOT use bullets, tables, lists, or meta text.\n\n"
        f"User question: {user_query}"
    )
    return (
        "### Instruction:\n"
        f"{instr.strip()}\n\n"
        "### Input:\n\n"
        "### Response:\n"
    )

# Test both prompt formats to see which works better
print("=== Testing Prompt Formats ===")
test_question = "I have a severe headache. What should I do?"
print("\n1. Current format (simple):")
print(create_prompt_simple(test_question))
print("\n2. Alpaca format:")
print(create_prompt_alpaca_style(test_question))
print("\n→ If your model was trained with Alpaca format, replace create_prompt() with create_prompt_alpaca_style()")


In [ ]:
# ============================================
# OPTION 2: Improved Generation Parameters
# ============================================
# Fine-tuned models often need different generation settings

# Option A: More relaxed settings (try if outputs are too short/repetitive)
GEN_KW_RELAXED = dict(
    max_new_tokens=200,        # Increased from 140
    do_sample=True,            # Enable sampling instead of deterministic
    temperature=0.7,          # Add some creativity
    top_p=0.9,                # Nucleus sampling
    num_beams=3,              # Reduced from 4
    early_stopping=True,
    no_repeat_ngram_size=2,   # Reduced from 3
    length_penalty=1.2        # Encourage longer responses
)

# Option B: Conservative settings (try if outputs are too verbose/wrong)
GEN_KW_CONSERVATIVE = dict(
    max_new_tokens=120,
    do_sample=False,
    num_beams=5,              # Increased from 4
    early_stopping=True,
    no_repeat_ngram_size=4,
    length_penalty=0.8        # Discourage overly long responses
)

# Option C: Current settings (baseline)
GEN_KW_CURRENT = dict(
    max_new_tokens=140,
    do_sample=False,
    num_beams=4,
    early_stopping=True,
    no_repeat_ngram_size=3,
    length_penalty=0.9
)

print("Generation parameter options available:")
print("1. GEN_KW_RELAXED - for short/repetitive outputs")
print("2. GEN_KW_CONSERVATIVE - for verbose/incorrect outputs")
print("3. GEN_KW_CURRENT - baseline settings")
print("\n→ Replace GEN_KW in generate_responses() with one of the above")


In [ ]:
# ============================================
# QUICK FIX: Try Alpaca Format + Improved Parameters
# ============================================
# This creates a modified version using Alpaca format (same as base model)

# Modified prompt function using Alpaca format
def create_prompt_finetuned_alpaca(user_query: str) -> str:
    """Use Alpaca format for fine-tuned model (matches base model format)."""
    instr = (
        "You are a careful, concise triage assistant for the public. "
        "Write ONE short paragraph of 3–6 sentences. Be practical and safe. "
        "Include immediate self-care, specific OTC dosing when appropriate, and clear red flags that require urgent care. "
        "Do NOT ask questions. Do NOT use bullets, tables, lists, or meta text.\n\n"
        f"User question: {user_query}"
    )
    return (
        "### Instruction:\n"
        f"{instr.strip()}\n\n"
        "### Input:\n\n"
        "### Response:\n"
    )

# Modified dataset class
class QADatasetFinetuned(Dataset):
    def __init__(self, questions, references, tokenizer, max_length=512, use_alpaca=True):
        self.questions = questions
        self.references = references
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.use_alpaca = use_alpaca  # Toggle between formats

    def __len__(self):
        return len(self.questions)

    def __getitem__(self, idx):
        q = self.questions[idx]
        # Use Alpaca format if enabled, otherwise use original
        if self.use_alpaca:
            prompt = create_prompt_finetuned_alpaca(q)
        else:
            prompt = create_prompt(q)  # Original format

        enc = self.tokenizer(prompt, max_length=self.max_length, padding="max_length",
                             truncation=True, return_tensors="pt")
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "prompt_length": enc["input_ids"].shape[1],
            "question": q,
            "reference": self.references[idx],
        }

print("✓ Created QADatasetFinetuned with format toggle")
print("→ Set use_alpaca=True to match base model format")
print("→ Set use_alpaca=False to use original fine-tuned format")


In [ ]:
# ============================================
# QUICK TEST: Compare Both Formats on One Question
# ============================================
# Run this to quickly see which format works better for your fine-tuned model

def quick_test_formats(model, tokenizer, test_question="I have a severe headache. What should I do?"):
    """Quick test to see which format produces better responses."""
    print(f"\n{'='*70}")
    print("QUICK TEST: Comparing Simple vs Alpaca Format")
    print(f"{'='*70}\n")
    print(f"Test Question: {test_question}\n")

    formats = [
        ("Simple Format", create_prompt_simple_finetuned, False),
        ("Alpaca Format", create_prompt_finetuned_alpaca, True)
    ]

    results = {}
    device = "cuda" if torch.cuda.is_available() else "cpu"

    for format_name, prompt_fn, use_alpaca in formats:
        print(f"\n{'─'*70}")
        print(f"Testing {format_name}")
        print(f"{'─'*70}")

        prompt = prompt_fn(test_question)
        print(f"\nPrompt:\n{prompt}\n")

        # Tokenize
        enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
        input_ids = enc["input_ids"].to(device)
        attention_mask = enc["attention_mask"].to(device)

        # Generate
        print("Generating response...")
        with torch.no_grad():
            gen = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=150,
                do_sample=False,
                num_beams=4,
                early_stopping=True,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id,
            )

        # Decode
        prompt_len = input_ids.shape[1]
        response = tokenizer.decode(gen[0][prompt_len:], skip_special_tokens=True).strip()
        results[format_name] = response

        print(f"\nResponse ({format_name}):")
        print(f"{response}\n")
        print(f"Length: {len(response)} characters")

    print(f"\n{'='*70}")
    print("COMPARISON SUMMARY")
    print(f"{'='*70}")
    print("\nCompare the responses above and choose the format that:")
    print("  ✓ Produces more accurate medical information")
    print("  ✓ Follows the instruction format better")
    print("  ✓ Generates appropriate length responses")
    print("  ✓ Avoids asking questions or using bullet points")
    print("\n💡 Typically, if your model was trained with Alpaca format,")
    print("   the Alpaca format will produce significantly better results!")

    return results

# Uncomment to run the quick test (recommended):
# test_results = quick_test_formats(model, tokenizer)


In [ ]:

# ============================================
# RECOMMENDED: Generate with Alpaca Format
# ============================================
# Run this cell to regenerate with Alpaca format (matches base model)

def generate_responses_with_alpaca(model, tokenizer, questions, references, gen_kwargs=None):
    """Generate responses using Alpaca format (same as base model)."""
    if gen_kwargs is None:
        gen_kwargs = GEN_KW_CURRENT  # or try GEN_KW_RELAXED

    ds = QADatasetFinetuned(questions, references, tokenizer, use_alpaca=True)
    dl = DataLoader(ds, batch_size=1, shuffle=False)

    ban_strings = ["Table", "Figure", "?"]
    bad_words_ids = []
    for s in ban_strings:
        ids = tokenizer.encode(s, add_special_tokens=False)
        if ids:
            bad_words_ids.append(ids)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    outs = []

    for i, batch in enumerate(dl, 1):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        prompt_len = batch["prompt_length"][0]
        question = batch["question"][0]
        reference = batch["reference"][0]

        print(f"\n{'='*60}\nSample {i}/{len(questions)} (Alpaca Format)\n{'='*60}")
        print("Question:", question)

        cm = torch.autocast("cuda", dtype=torch.float16) if torch.cuda.is_available() else nullcontext()
        with torch.no_grad(), cm:
            gen = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id,
                bad_words_ids=bad_words_ids if bad_words_ids else None,
                **gen_kwargs
            )

        new_tokens = gen[0, prompt_len:] if gen.shape[1] > prompt_len else gen[0]
        raw = tokenizer.decode(new_tokens, skip_special_tokens=True)
        cleaned = validate_answer_paragraph(raw)

        print("\n🤖 Generated (Alpaca Format):\n", cleaned)
        print("\n✅ Reference:\n", reference)
        outs.append(cleaned)

    print(f"\n✅ Successfully processed {len(outs)} samples with Alpaca format")
    return outs

# Uncomment to run:
# generated_outputs_ft_alpaca = generate_responses_with_alpaca(
#     model, tokenizer, questions, references, gen_kwargs=GEN_KW_RELAXED
# )
print("✓ Function ready. Uncomment the last two lines to generate with Alpaca format.")


In [ ]:
import re
import torch
from torch.utils.data import DataLoader, Dataset
from contextlib import nullcontext

# ---------- Prompt ----------
def create_prompt(user_query: str) -> str:
    return (
        "You are a careful, concise triage assistant for the public.\n"
        "Write ONE short paragraph (3–6 sentences). Be direct, practical, and safe.\n"
        "Include: immediate self-care, specific OTC dosing if appropriate, and clear red flags for urgent care.\n"
        "Rules: Do NOT ask the user any questions. Do NOT use bullet points, tables, or meta text.\n\n"
        f"User: {user_query}\n"
        "Answer (one short paragraph, no questions): "
    )

# ---------- Cleaner ----------
def validate_answer_paragraph(text: str) -> str:
    # Remove meta words and extra spaces
    text = re.sub(r"(Table|Figure|Assistant:|as shown in|refer to)", "", text, flags=re.I).strip()
    # Split into sentences naïvely; drop question-like sentences
    # Replace common line breaks with spaces to avoid accidental splits
    text = re.sub(r"\s*\n+\s*", " ", text)
    # Simple sentence split by period. Keep punctuation in next step if present.
    raw_sents = re.split(r"(?<=[\.\!\?])\s+", text)
    # Drop questions or empty bits
    q_like = re.compile(r"\?\s*$")
    sents = [s.strip() for s in raw_sents if s.strip() and not q_like.search(s)]
    # Cap to 3–6 sentences
    sents = sents[:6]
    if len(sents) < 3 and "." not in text:
        # As fallback, force a single trimmed paragraph without '?'
        text = re.sub(r"\?", "", text)
        return text.strip()
    # Ensure each sentence ends with a period if missing
    sents = [s if re.search(r"[\.!\)]\s*$", s) else s + "." for s in sents]
    return " ".join(sents)

# ---------- Minimal QADataset for paragraph prompting ----------
class QADataset(Dataset):
    def __init__(self, questions, references, tokenizer, max_length=512):
        self.questions = questions
        self.references = references
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self): return len(self.questions)
    def __getitem__(self, idx):
        q = self.questions[idx]
        prompt = create_prompt(q)
        enc = self.tokenizer(prompt, max_length=self.max_length, padding="max_length",
                             truncation=True, return_tensors="pt")
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "prompt_length": enc["input_ids"].shape[1],
            "question": q,
            "reference": self.references[idx],
        }

# ---------- Deterministic decoding; ban '?' to prevent questions ----------
GEN_KW = dict(
    max_new_tokens=140,
    do_sample=False,          # deterministic
    num_beams=4,
    early_stopping=True,
    no_repeat_ngram_size=3,
    length_penalty=0.9
)

def generate_responses(model, tokenizer, questions, references):
    ds = QADataset(questions, references, tokenizer)
    dl = DataLoader(ds, batch_size=1, shuffle=False)

    # Build bad_words_ids and include "?" token if present
    ban_strings = ["Table", "Figure", "?"]
    bad_words_ids = []
    for s in ban_strings:
        ids = tokenizer.encode(s, add_special_tokens=False)
        if ids:
            bad_words_ids.append(ids)  # list[list[int]]

    device = "cuda" if torch.cuda.is_available() else "cpu"
    outs = []

    for i, batch in enumerate(dl, 1):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        prompt_len = batch["prompt_length"][0]
        question = batch["question"][0]
        reference = batch["reference"][0]

        print(f"\n{'='*60}\nSample {i}/{len(questions)}\n{'='*60}")
        print("Question:", question)

        cm = torch.autocast("cuda", dtype=torch.float16) if torch.cuda.is_available() else nullcontext()
        with torch.no_grad(), cm:
            gen = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id,
                bad_words_ids=bad_words_ids if bad_words_ids else None,
                **GEN_KW
            )

        new_tokens = gen[0, prompt_len:] if gen.shape[1] > prompt_len else gen[0]
        raw = tokenizer.decode(new_tokens, skip_special_tokens=True)
        cleaned = validate_answer_paragraph(raw)

        print("\nGenerated:\n", cleaned)
        print("\nReference:\n", reference)
        outs.append(cleaned)

    print(f"\nSuccessfully processed {len(outs)} samples")
    return outs

print("Starting response generation...")
generated_outputs = generate_responses(model, tokenizer, questions, references)
print("Done.")






In [ ]:
import os
# Set your OpenAI API key from environment variable for security
# os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")  # Get from environment variable
# Or set it manually (NOT RECOMMENDED for production):
# os.environ["OPENAI_API_KEY"] = "your-api-key-here"

In [ ]:
# ==========================
# OpenAI setup (NEW SDK) + helpers
# ==========================
import os, re
import numpy as np
from nltk.tokenize import word_tokenize
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from bert_score import score as bert_score
from sentence_transformers import util
from torch import cuda
from openai import OpenAI

# ---- Get API key from environment variable (secure) ----
# Make sure to set OPENAI_API_KEY environment variable before running
# os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# New SDK client (reads OPENAI_API_KEY from environment)
client = OpenAI()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = word_tokenize(text)
    return " ".join(tokens)

def compute_bleu_score(generated, reference):
    gen_tokens = word_tokenize(generated)
    ref_tokens = word_tokenize(reference)
    smoothie = SmoothingFunction().method4
    return sentence_bleu([ref_tokens], gen_tokens, smoothing_function=smoothie)

def compute_hybrid_score(bert_f1, bleu, bert_weight=0.7):
    return bert_weight * bert_f1 + (1 - bert_weight) * bleu

def call_gpt4o(prompt, max_tokens=10):
    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2,
            max_tokens=max_tokens,
        )
        reply = response.choices[0].message.content.strip()
        match = re.search(r"\d*\.\d+|\d+", reply)
        return float(match.group(0)) if match else 0.5
    except Exception as e:
        print(f"GPT-4o error: {e}")
        return 0.5

def compute_llm_judge_score(generated, reference, question):
    prompt = (
        "You are an expert medical evaluator. Score the following generated answer from 0 to 1 "
        "based on factual accuracy and clinical relevance to the reference.\n\n"
        f"Question: {question}\n"
        f"Generated Answer: {generated}\n"
        f"Reference Answer: {reference}\n\n"
        "Just reply with a score (e.g., 0.73)."
    )
    return call_gpt4o(prompt)

def compute_geval_score(generated, reference, question):
    prompt = (
        "You are evaluating the clinical quality of a generated answer. Consider factual correctness, "
        "completeness, and clarity. Score it from 0 to 1.\n\n"
        f"Question: {question}\n"
        f"Generated Answer: {generated}\n"
        f"Reference Answer: {reference}\n\n"
        "Reply with a score (e.g., 0.82)."
    )
    return call_gpt4o(prompt)

def compute_metrics_per_query(generated_outputs, references, questions, nlp, embedder):
    rouge_scorer_obj = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

    # Metric containers
    bert_p_scores, bert_r_scores, bert_f1_scores = [], [], []
    rouge_l_scores, fcs_scores, bleu_scores, hybrid_scores = [], [], [], []
    entity_f1_scores, mcr_scores, meteor_scores = [], [], []
    llm_judge_scores, geval_scores = [], []

    print(f"\n=== Per-Query Evaluation Metrics ===")

    for i, (gen, ref, question) in enumerate(zip(generated_outputs, references, questions), 1):
        gen_norm = preprocess_text(gen)
        ref_norm = preprocess_text(ref)

        # BERTScore
        p, r, f1 = bert_score([gen_norm], [ref_norm], lang="en", model_type="roberta-large")
        bert_p, bert_r, bert_f1 = p.item(), r.item(), f1.item()

        # ROUGE-L
        rouge_l = rouge_scorer_obj.score(ref_norm, gen_norm)['rougeL'].fmeasure

        # Entity F1
        gen_entities = {ent.text.lower() for ent in nlp(gen).ents if ent.label_ in ["DISEASE", "CHEMICAL"]}
        ref_entities = {ent.text.lower() for ent in nlp(ref).ents if ent.label_ in ["DISEASE", "CHEMICAL"]}
        if ref_entities:
            precision = len(gen_entities & ref_entities) / len(gen_entities) if gen_entities else 0
            recall = len(gen_entities & ref_entities) / len(ref_entities)
            entity_f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        else:
            entity_f1 = 1.0 if not gen_entities else 0.0

        # FCS (Fact Checking Score via cosine similarity)
        dev = "cuda" if cuda.is_available() else "cpu"
        gen_embedding = embedder.encode(gen, convert_to_tensor=True, device=dev)
        ref_embedding = embedder.encode(ref, convert_to_tensor=True, device=dev)
        fcs = util.cos_sim(gen_embedding, ref_embedding)[0][0].item()

        # MCR (Medical Concept Recall)
        mcr = (len(gen_entities & ref_entities) / len(ref_entities)) if ref_entities else (1.0 if not gen_entities else 0.0)

        # BLEU, METEOR, Hybrid
        bleu = compute_bleu_score(gen_norm, ref_norm)
        meteor = meteor_score([word_tokenize(ref_norm)], word_tokenize(gen_norm))
        hybrid_score = compute_hybrid_score(bert_f1, bleu)

        # LLM-based scores
        llm_judge_score = compute_llm_judge_score(gen, ref, question)
        geval_score = compute_geval_score(gen, ref, question)

        # Store
        bert_p_scores.append(bert_p)
        bert_r_scores.append(bert_r)
        bert_f1_scores.append(bert_f1)
        rouge_l_scores.append(rouge_l)
        entity_f1_scores.append(entity_f1)
        fcs_scores.append(fcs)
        mcr_scores.append(mcr)
        bleu_scores.append(bleu)
        meteor_scores.append(meteor)
        hybrid_scores.append(hybrid_score)
        llm_judge_scores.append(llm_judge_score)
        geval_scores.append(geval_score)

        # Print per-sample summary
        print(f"\nSample {i}: {question}")
        print(f"Generated Answer: {gen}")
        print(f"Reference Answer: {ref}")
        print(f"BERTScore P/R/F1: {bert_p:.4f}/{bert_r:.4f}/{bert_f1:.4f}")
        print(f"ROUGE-L: {rouge_l:.4f} | FCS: {fcs:.4f} | BLEU: {bleu:.4f} | METEOR: {meteor:.4f} | Hybrid: {hybrid_score:.4f}")
        print(f"Entity F1: {entity_f1:.4f} | MCR: {mcr:.4f}")
        print(f"LLM Judge (GPT-4o): {llm_judge_score:.4f} | GEval (GPT-4o): {geval_score:.4f}")

    # Averages
    print("\n=== Average Metrics Across All Queries ===")
    print(f"Average BERT-P: {np.mean(bert_p_scores):.4f}")
    print(f"Average BERT-R: {np.mean(bert_r_scores):.4f}")
    print(f"Average BERT-F1: {np.mean(bert_f1_scores):.4f}")
    print(f"Average ROUGE-L: {np.mean(rouge_l_scores):.4f}")
    print(f"Average FCS: {np.mean(fcs_scores):.4f}")
    print(f"Average Entity F1: {np.mean(entity_f1_scores):.4f}")
    print(f"Average MCR: {np.mean(mcr_scores):.4f}")
    print(f"Average BLEU: {np.mean(bleu_scores):.4f}")
    print(f"Average METEOR: {np.mean(meteor_scores):.4f}")
    print(f"Average Hybrid: {np.mean(hybrid_scores):.4f}")
    print(f"Average LLM Judge (GPT-4o): {np.mean(llm_judge_scores):.4f}")
    print(f"Average GEval (GPT-4o): {np.mean(geval_scores):.4f}")


In [ ]:
import nltk
nltk.download('punkt_tab')

In [ ]:
# Evaluate generated outputs
if generated_outputs:
    compute_metrics_per_query(generated_outputs, references, questions, nlp, embedder)
else:
    print("No outputs generated due to error.")

# Clear GPU memory
torch.cuda.empty_cache()

In [ ]:
# Preserve your fine-tuned run outputs
generated_outputs_ft = list(generated_outputs)  # from your previous run


In [ ]:
# ===== A) Build per-sample DataFrame + AVG (works for FT or BASE) =====
import numpy as np
import pandas as pd

def build_metrics_df(gens, refs, qs, nlp, embedder, bert_model="roberta-large", use_llm=True):
    from rouge_score import rouge_scorer
    from bert_score import score as bert_score
    from nltk.tokenize import word_tokenize
    from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu
    from nltk.translate.meteor_score import meteor_score
    from sentence_transformers import util
    from torch import cuda

    assert len(gens) == len(refs) == len(qs), "gens/refs/qs must have equal length"
    rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    dev = "cuda" if cuda.is_available() else "cpu"
    smooth = SmoothingFunction().method4

    rows = []
    for i, (gen, ref, q) in enumerate(zip(gens, refs, qs), 1):
        gen_norm, ref_norm = preprocess_text(gen), preprocess_text(ref)

        # BERTScore
        p, r, f1 = bert_score([gen_norm], [ref_norm], lang="en", model_type=bert_model)
        bert_p, bert_r, bert_f1 = p.item(), r.item(), f1.item()

        # ROUGE-L
        rouge_l = rouge.score(ref_norm, gen_norm)['rougeL'].fmeasure

        # Entities (DISEASE, CHEMICAL) + MCR
        if nlp:
            gen_ents = {e.text.lower() for e in nlp(gen).ents if e.label_ in ["DISEASE", "CHEMICAL"]}
            ref_ents = {e.text.lower() for e in nlp(ref).ents if e.label_ in ["DISEASE", "CHEMICAL"]}
        else:
            gen_ents, ref_ents = set(), set()
        if ref_ents:
            inter = len(gen_ents & ref_ents)
            prec = inter / len(gen_ents) if gen_ents else 0.0
            rec  = inter / len(ref_ents)
            entity_f1 = 2*prec*rec/(prec+rec) if (prec+rec) else 0.0
            mcr = inter / len(ref_ents)
        else:
            entity_f1 = 1.0 if not gen_ents else 0.0
            mcr = 1.0 if not gen_ents else 0.0

        # FCS
        if embedder:
            g_emb = embedder.encode(gen, convert_to_tensor=True, device=dev)
            r_emb = embedder.encode(ref, convert_to_tensor=True, device=dev)
            fcs = util.cos_sim(g_emb, r_emb)[0][0].item()
        else:
            fcs = np.nan

        # BLEU / METEOR / Hybrid
        bleu = sentence_bleu([word_tokenize(ref_norm)], word_tokenize(gen_norm), smoothing_function=smooth)
        meteor_val = meteor_score([word_tokenize(ref_norm)], word_tokenize(gen_norm))
        hybrid = compute_hybrid_score(bert_f1, bleu, bert_weight=0.7)

        # LLM judges (optional)
        if use_llm:
            llm_judge = compute_llm_judge_score(gen, ref, q)
            geval = compute_geval_score(gen, ref, q)
        else:
            llm_judge = np.nan
            geval = np.nan

        rows.append({
            "sample": i, "question": q, "generated": gen, "reference": ref,
            "bert_p": bert_p, "bert_r": bert_r, "bert_f1": bert_f1,
            "rouge_l": rouge_l, "fcs": fcs,
            "entity_f1": entity_f1, "mcr": mcr,
            "bleu": bleu, "meteor": meteor_val,
            "hybrid": hybrid, "llm_judge": llm_judge, "geval": geval,
        })

    df = pd.DataFrame(rows)
    # append AVG row (numeric metrics only)
    avg = df.drop(columns=["sample","question","generated","reference"], errors="ignore").mean(numeric_only=True)
    avg_row = {"sample": "AVG", "question": ""}
    avg_row.update(avg.to_dict())
    df = pd.concat([df, pd.DataFrame([avg_row])], ignore_index=True)
    return df


In [ ]:
# Build DF for FINETUNED using your existing outputs
df_ft = build_metrics_df(generated_outputs, references, questions, nlp, embedder, use_llm=True)  # set False to skip GPT calls
display(df_ft)


**BASE MODEL**

In [ ]:
# ============================================
# 0) Config & Imports
# ============================================
import os, re, numpy as np, pandas as pd, torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from nltk.tokenize import word_tokenize
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from bert_score import score as bert_score
from sentence_transformers import util
from torch import cuda

# ---- Set this True to call GPT judges (scores never NaN; fallback=0.5 on error)
DO_LLM_JUDGES = True

# ---- Filenames
BASE_METRICS_CSV = "medallama_base_metrics.csv"
BASE_AVG_CSV     = "medallama_base_metrics_avg.csv"

# ---- Decoding (deterministic, overlap-friendly)
GEN_KW_BASE = dict(
    max_new_tokens=120,
    do_sample=False,
    num_beams=6,
    early_stopping=True,
    no_repeat_ngram_size=4,
    length_penalty=1.0
)

# Reproducibility
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Pull optional deps from globals if you already loaded them elsewhere
nlp      = globals().get("nlp", None)          # spaCy NER (e.g., en_ner_bc5cdr_md)
embedder = globals().get("embedder", None)     # SentenceTransformer instance


In [ ]:
# ============================================
# 1) Optional: OpenAI Judges (robust, no NaN)
# ============================================
# Ensure OPENAI_API_KEY is in env (don't hardcode secrets)
# os.environ["OPENAI_API_KEY"] = "sk-..."  # if you must, but env var is preferred

from openai import OpenAI
_client = None

def _get_openai_client():
    global _client
    if _client is None:
        try:
            _client = OpenAI()
        except Exception:
            _client = None
    return _client

def _call_gpt4o(prompt, max_tokens=10):
    """
    Robust judge call: returns a float; falls back to 0.5 on any exception.
    This guarantees no NaN in outputs.
    """
    try:
        client = _get_openai_client()
        if client is None:
            return 0.5
        r = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role":"user","content":prompt}],
            temperature=0.2,
            max_tokens=max_tokens,
        )
        reply = r.choices[0].message.content.strip()
        m = re.search(r"\d*\.\d+|\d+", reply)
        return float(m.group(0)) if m else 0.5
    except Exception:
        return 0.5

def compute_llm_judge_score(generated, reference, question):
    if not DO_LLM_JUDGES: return 0.5
    prompt = (
        "You are an expert medical evaluator. Score the following generated answer from 0 to 1 "
        "based on factual accuracy and clinical relevance to the reference.\n\n"
        f"Question: {question}\n"
        f"Generated Answer: {generated}\n"
        f"Reference Answer: {reference}\n\n"
        "Just reply with a score (e.g., 0.73)."
    )
    return _call_gpt4o(prompt)

def compute_geval_score(generated, reference, question):
    if not DO_LLM_JUDGES: return 0.5
    prompt = (
        "You are evaluating the clinical quality of a generated answer. Consider factual correctness, "
        "completeness, and clarity. Score it from 0 to 1.\n\n"
        f"Question: {question}\n"
        f"Generated Answer: {generated}\n"
        f"Reference Answer: {reference}\n\n"
        "Reply with a score (e.g., 0.82)."
    )
    return _call_gpt4o(prompt)


In [ ]:
# --------- Load BASE MedLLaMA ------------
def load_base_medllama(model_name="meta-llama/Meta-Llama-3-8B"):
    tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    use_4bit = False
    try:
        import bitsandbytes as bnb  # noqa
        use_4bit = torch.cuda.is_available()
    except Exception:
        use_4bit = False

    if use_4bit:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=False,
        )
        mdl = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            torch_dtype=torch.float16,
            quantization_config=bnb_config,
            trust_remote_code=True,
            low_cpu_mem_usage=True,
        )
    else:
        dtype = torch.float16 if torch.cuda.is_available() else torch.float32
        mdl = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto" if torch.cuda.is_available() else None,
            torch_dtype=dtype if torch.cuda.is_available() else None,
            trust_remote_code=True,
            low_cpu_mem_usage=True,
        )

    mdl.config.pad_token_id = tok.pad_token_id
    print("✅ Loaded base:", model_name)
    return mdl, tok

base_model, base_tok = load_base_medllama("meta-llama/Meta-Llama-3-8B")

In [ ]:
# ---------- Prompt template for MedLLaMA (ALPACA-style) ----------
def alpaca_prompt(instruction: str, input_text: str = "") -> str:
    return (
        "### Instruction:\n"
        f"{instruction.strip()}\n\n"
        "### Input:\n"
        f"{input_text.strip()}\n\n"
        "### Response:\n"
    )

def create_prompt_for_base(user_query: str) -> str:
    instr = (
        "You are a careful, concise triage assistant for the public. "
        "Write ONE short paragraph of 3–6 sentences. Be practical and safe. "
        "Include immediate self-care, specific OTC dosing when appropriate, and clear red flags that require urgent care. "
        "Do NOT ask questions. Do NOT use bullets, tables, lists, or meta text.\n\n"
        f"User question: {user_query}"
    )
    return alpaca_prompt(instr, input_text="")

def validate_answer_paragraph(text: str) -> str:
    # Remove meta/noise
    text = re.sub(r"(Table|Figure|Assistant:|as shown in|refer to)", "", text, flags=re.I)
    # One paragraph, no questions
    text = re.sub(r"\s*\n+\s*", " ", text).strip()
    sents = re.split(r"(?<=[\.\!\?])\s+", text)
    sents = [s for s in sents if s and not s.strip().endswith("?")]
    sents = sents[:6]
    if not sents:
        return "Use rest, hydration, appropriate OTC dosing, and seek urgent care if severe or high-risk features occur."
    sents = [s if re.search(r"[\.!\)]\s*$", s) else s + "." for s in sents]
    return " ".join(sents)

# ---------- Dataset / Generator ----------
class QADatasetBase(Dataset):
    def __init__(self, questions, references, tokenizer, max_length=512):
        self.questions = questions
        self.references = references
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self): return len(self.questions)
    def __getitem__(self, idx):
        q = self.questions[idx]
        prompt = create_prompt_for_base(q)
        enc = self.tokenizer(
            prompt, max_length=self.max_length, padding="max_length",
            truncation=True, return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "prompt_length": enc["input_ids"].shape[1],
            "question": q,
            "reference": self.references[idx],
        }

# Deterministic decoding to maximize lexical overlap
GEN_KW_BASE = dict(
    max_new_tokens=100,
    do_sample=False,
    num_beams=6,
    early_stopping=True,
    no_repeat_ngram_size=4,
    length_penalty=1.0
)

def generate_responses_base(model, tokenizer, questions, references):
    ds = QADatasetBase(questions, references, tokenizer)
    dl = DataLoader(ds, batch_size=1, shuffle=False)

    # Ban a few strings (including '?') to discourage questions
    ban_strings = ["Table", "Figure", "?"]
    bad_words_ids = []
    for s in ban_strings:
        ids = tokenizer.encode(s, add_special_tokens=False)
        if ids:
            bad_words_ids.append(ids)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    outs = []

    for i, batch in enumerate(dl, 1):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        prompt_len = batch["prompt_length"][0]

        print("\n" + "="*60)
        print(f"Sample {i}/{len(questions)}")
        print("="*60)
        print("Question:", batch["question"][0])

        with torch.no_grad():
            gen = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id,
                bad_words_ids=bad_words_ids if bad_words_ids else None,
                **GEN_KW_BASE
            )

        new_tokens = gen[0, prompt_len:] if gen.shape[1] > prompt_len else gen[0]
        raw = tokenizer.decode(new_tokens, skip_special_tokens=True)
        cleaned = validate_answer_paragraph(raw)

        print("\n🤖 Generated:\n", cleaned)
        print("\n✅ Reference:\n", batch["reference"][0])
        outs.append(cleaned)

    print(f"\n✅ Successfully processed {len(outs)} samples (base)")
    return outs

print("✅ Prompt & generator ready")


In [ ]:
# ============================================
# 4) Metrics (per-sample print + DF + AVG)
# ============================================
def _normalize_for_overlap(s: str):
    s = s.lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return " ".join(word_tokenize(s))

def compute_hybrid_score(bert_f1, bleu, bert_weight=0.7):
    return bert_weight * bert_f1 + (1 - bert_weight) * bleu

def evaluate_and_print_per_sample(
    gens, refs, qs, nlp=None, embedder=None, bert_model="roberta-large"
):
    assert len(gens) == len(refs) == len(qs), "gens/refs/qs must have equal length"
    rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    dev = "cuda" if cuda.is_available() else "cpu"
    smooth = SmoothingFunction().method4

    rows = []
    print("\n=== Per-Query Evaluation Metrics (BASE) ===")
    for i, (gen, ref, q) in enumerate(zip(gens, refs, qs), 1):
        gen_norm, ref_norm = _normalize_for_overlap(gen), _normalize_for_overlap(ref)

        # BERTScore
        p, r, f1 = bert_score([gen_norm], [ref_norm], lang="en", model_type=bert_model)
        bert_p, bert_r, bert_f1 = p.item(), r.item(), f1.item()

        # ROUGE-L
        rouge_l = rouge.score(ref_norm, gen_norm)['rougeL'].fmeasure

        # Entities + MCR
        if nlp:
            gen_ents = {e.text.lower() for e in nlp(gen).ents if e.label_ in ["DISEASE", "CHEMICAL"]}
            ref_ents = {e.text.lower() for e in nlp(ref).ents if e.label_ in ["DISEASE", "CHEMICAL"]}
        else:
            gen_ents, ref_ents = set(), set()
        if ref_ents:
            inter = len(gen_ents & ref_ents)
            prec = inter / len(gen_ents) if gen_ents else 0.0
            rec  = inter / len(ref_ents)
            entity_f1 = 2*prec*rec/(prec+rec) if (prec+rec) else 0.0
            mcr = inter / len(ref_ents)
        else:
            entity_f1 = 1.0 if not gen_ents else 0.0
            mcr = 1.0 if not gen_ents else 0.0

        # FCS
        if embedder:
            g_emb = embedder.encode(gen, convert_to_tensor=True, device=dev)
            r_emb = embedder.encode(ref, convert_to_tensor=True, device=dev)
            fcs = util.cos_sim(g_emb, r_emb)[0][0].item()
        else:
            fcs = float("nan")

        # BLEU / METEOR
        bleu = sentence_bleu([word_tokenize(ref_norm)], word_tokenize(gen_norm), smoothing_function=smooth)
        meteor_val = meteor_score([word_tokenize(ref_norm)], word_tokenize(gen_norm))

        # Hybrid
        hybrid = compute_hybrid_score(bert_f1, bleu, bert_weight=0.7)

        # LLM judges (robust; never NaN)
        llm_judge = compute_llm_judge_score(gen, ref, q)
        geval     = compute_geval_score(gen, ref, q)

        # Store row
        rows.append({
            "sample": i, "question": q,
            "generated": gen, "reference": ref,
            "bert_p": bert_p, "bert_r": bert_r, "bert_f1": bert_f1,
            "rouge_l": rouge_l, "fcs": fcs,
            "entity_f1": entity_f1, "mcr": mcr,
            "bleu": bleu, "meteor": meteor_val,
            "hybrid": hybrid, "llm_judge": llm_judge, "geval": geval,
        })

        # Print per-sample
        print(f"\nSample {i}: {q}")
        print(f"Generated Answer: {gen}")
        print(f"Reference Answer: {ref}")
        print(f"BERTScore P/R/F1: {bert_p:.4f}/{bert_r:.4f}/{bert_f1:.4f}")
        print(f"ROUGE-L: {rouge_l:.4f} | FCS: {fcs if not np.isnan(fcs) else 0.0:.4f} | BLEU: {bleu:.4f} | METEOR: {meteor_val:.4f} | Hybrid: {hybrid:.4f}")
        print(f"Entity F1: {entity_f1:.4f} | MCR: {mcr:.4f}")
        print(f"LLM Judge: {llm_judge:.4f} | GEval: {geval:.4f}")

    # Build DF + AVG row
    df = pd.DataFrame(rows)
    avg = df.drop(columns=["sample","question","generated","reference"], errors="ignore").mean(numeric_only=True)
    avg_row = {"sample": "AVG", "question": ""}
    avg_row.update(avg.to_dict())
    df = pd.concat([df, pd.DataFrame([avg_row])], ignore_index=True)

    # Print averages
    print("\n=== Average Metrics Across All Queries (BASE) ===")
    for k in ["bert_p","bert_r","bert_f1","rouge_l","fcs","entity_f1","mcr","bleu","meteor","hybrid","llm_judge","geval"]:
        if k in avg:
            val = avg[k]
            if np.isnan(val): val = 0.0
            print(f"Average {k.upper().replace('_','-')}: {val:.4f}")

    return df


In [ ]:
# ============================================
# 5) Run: Generate → Evaluate → Save
# ============================================

# You must define these lists earlier in your notebook:
try:
    questions, references
except NameError:
    raise RuntimeError("Please define `questions` and `references` lists first.")

# Generate
print("Starting base-model generation (Llama template)…")
generated_outputs_base = generate_responses_base(base_model, base_tok, questions, references)
print("Base-model generation done.")

# Evaluate (prints per-sample + averages; LLM judges enabled if DO_LLM_JUDGES=True)
df_base = evaluate_and_print_per_sample(
    generated_outputs_base,
    references,
    questions,
    nlp=nlp,
    embedder=embedder,
    bert_model="roberta-large",
)

# Display + Save
pd.set_option("display.max_colwidth", 140)
display(df_base)

df_base.to_csv(BASE_METRICS_CSV, index=False)
base_avg = df_base[df_base["sample"].astype(str).str.strip().str.upper() == "AVG"].copy()
if base_avg.empty:
    metrics_only = df_base.drop(columns=["sample","question","generated","reference"], errors="ignore")
    base_avg = metrics_only.mean(numeric_only=True).to_frame().T
    base_avg.insert(0, "sample", "AVG")
    base_avg.insert(1, "question", "")
base_avg.round(4).to_csv(BASE_AVG_CSV, index=False)

print(f"\n✓ Saved BASE full metrics to: {BASE_METRICS_CSV}")
print(f"✓ Saved BASE AVG row to:      {BASE_AVG_CSV}")


In [ ]:
print("Starting base-model generation (Alpaca template)...")
generated_outputs_base = generate_responses_base(base_model, base_tok, questions, references)
print("Base-model generation done.")


In [ ]:
def _normalize_for_overlap(s: str):
    s = s.lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def compute_all_metrics_as_df(gens, refs, qs, nlp, embedder, do_llm_judges=False):
    rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    rows = []

    for i, (gen, ref, q) in enumerate(zip(gens, refs, qs), 1):
        gen_norm, ref_norm = _normalize_for_overlap(gen), _normalize_for_overlap(ref)

        # BERTScore
        p, r, f1 = bert_score([gen_norm], [ref_norm], lang="en", model_type="roberta-large")
        bert_p, bert_r, bert_f = p.item(), r.item(), f1.item()

        # ROUGE-L
        rouge_l = rouge.score(ref_norm, gen_norm)['rougeL'].fmeasure

        # Entity F1 + MCR (DISEASE, CHEMICAL)
        gen_ents = {e.text.lower() for e in (nlp(gen).ents if nlp else []) if e.label_ in ["DISEASE", "CHEMICAL"]}
        ref_ents = {e.text.lower() for e in (nlp(ref).ents if nlp else []) if e.label_ in ["DISEASE", "CHEMICAL"]}
        if ref_ents:
            inter = len(gen_ents & ref_ents)
            prec = inter / len(gen_ents) if gen_ents else 0.0
            rec = inter / len(ref_ents)
            ent_f1 = 2*prec*rec/(prec+rec) if (prec+rec) else 0.0
            mcr = inter / len(ref_ents)
        else:
            ent_f1 = 1.0 if not gen_ents else 0.0
            mcr = 1.0 if not gen_ents else 0.0

        # FCS
        dev = "cuda" if cuda.is_available() else "cpu"
        g_emb = embedder.encode(gen, convert_to_tensor=True, device=dev) if embedder else None
        r_emb = embedder.encode(ref, convert_to_tensor=True, device=dev) if embedder else None
        fcs = util.cos_sim(g_emb, r_emb)[0][0].item() if (g_emb is not None and r_emb is not None) else np.nan

        # BLEU / METEOR
        smooth = SmoothingFunction().method4
        bleu = sentence_bleu([word_tokenize(ref_norm)], word_tokenize(gen_norm), smoothing_function=smooth)
        meteor_val = meteor_score([word_tokenize(ref_norm)], word_tokenize(gen_norm))

        rows.append({
            "sample": i, "question": q,
            "bert_p": bert_p, "bert_r": bert_r, "bert_f1": bert_f,
            "rouge_l": rouge_l, "fcs": fcs,
            "entity_f1": ent_f1, "mcr": mcr,
            "bleu": bleu, "meteor": meteor_val,
        })

    df = pd.DataFrame(rows)
    avg = df.mean(numeric_only=True)
    avg_row = {"sample": "AVG", "question": ""}
    avg_row.update(avg.to_dict())
    df = pd.concat([df, pd.DataFrame([avg_row])], ignore_index=True)
    return df

df_base = compute_all_metrics_as_df(generated_outputs_base, references, questions, nlp, embedder)
pd.set_option("display.max_colwidth", 140)
display(df_base)

# Save
out_csv = "/content/medllama_base_metrics.csv"
df_base.to_csv(out_csv, index=False)
print("Saved metrics CSV to:", out_csv)


In [ ]:
# If you already have fine-tuned outputs in generated_outputs_ft, enable this block:

def add_suffix(df, suf):
    keep = ["sample", "question"]
    out = df.copy()
    out.rename(columns={c: f"{c}{suf}" for c in df.columns if c not in keep}, inplace=True)
    return out

try:
    generated_outputs_ft  # <- exists?
    df_ft = compute_all_metrics_as_df(generated_outputs_ft, references, questions, nlp, embedder)

    ft_s   = add_suffix(df_ft, "_ft")
    base_s = add_suffix(df_base, "_base")
    cmp_df = ft_s.merge(base_s.drop(columns=["question"]), on="sample", how="outer")

    col_order = ["sample", "question",
                 "bert_f1_ft", "bert_f1_base",
                 "rouge_l_ft", "rouge_l_base",
                 "bleu_ft", "bleu_base",
                 "meteor_ft", "meteor_base",
                 "entity_f1_ft", "entity_f1_base",
                 "mcr_ft", "mcr_base",
                 "fcs_ft", "fcs_base"]
    cols = [c for c in col_order if c in cmp_df.columns] + [c for c in cmp_df.columns if c not in col_order]
    cmp_df = cmp_df[cols]

    display(cmp_df)
    cmp_path = "/content/medllama_ft_vs_base_metrics.csv"
    cmp_df.to_csv(cmp_path, index=False)
    print("Saved comparison CSV to:", cmp_path)
except NameError:
    print("Note: 'generated_outputs_ft' not found. Skipping FT vs Base comparison.")


In [ ]:
def save_avg_row(df: pd.DataFrame, out_csv: str) -> pd.DataFrame:
    avg = df[df.get("sample", "").astype(str).str.strip().str.upper() == "AVG"].copy()
    if avg.empty:
        metrics_only = df.drop(columns=["sample","question","generated","reference"], errors="ignore")
        avg = metrics_only.mean(numeric_only=True).to_frame().T
        avg.insert(0, "sample", "AVG")
        avg.insert(1, "question", "")
    avg = avg.round(4)
    avg.to_csv(out_csv, index=False)
    print(f"✓ Saved AVG row to: {out_csv}")
    return avg


In [ ]:
FT_AVG_CSV = "medallama_ft_avg.csv"
avg_ft = save_avg_row(df_ft, FT_AVG_CSV)

In [ ]:
BASE_AVG_CSV = "/content/medallama_base_avg.csv"
avg_base = save_avg_row(df_base, BASE_AVG_CSV)


In [ ]:
def _suffix_columns(df: pd.DataFrame, suffix: str) -> pd.DataFrame:
    df = df.copy()
    for c in ["sample","question","generated","reference"]:
        if c in df.columns:
            df.drop(columns=[c], inplace=True)
    df.columns = [f"{c}{suffix}" for c in df.columns]
    return df

def compare_avgs(base_avg_csv: str, ft_avg_csv: str, out_csv: str) -> pd.DataFrame:
    base = pd.read_csv(base_avg_csv)
    ft   = pd.read_csv(ft_avg_csv)

    base_s = _suffix_columns(base, "_base")
    ft_s   = _suffix_columns(ft, "_ft")

    cmp_df = pd.concat([base_s, ft_s], axis=1)

    # nice column order if present
    preferred = [
        "bert_f1_base","bert_f1_ft",
        "rouge_l_base","rouge_l_ft",
        "bleu_base","bleu_ft",
        "meteor_base","meteor_ft",
        "entity_f1_base","entity_f1_ft",
        "mcr_base","mcr_ft",
        "fcs_base","fcs_ft",
        "hybrid_base","hybrid_ft",
        "llm_judge_base","llm_judge_ft",
        "geval_base","geval_ft",
        "bert_p_base","bert_p_ft",
        "bert_r_base","bert_r_ft",
    ]
    cols = [c for c in preferred if c in cmp_df.columns] + \
           [c for c in cmp_df.columns if c not in preferred]
    cmp_df = cmp_df[cols].round(4)

    cmp_df.to_csv(out_csv, index=False)
    print(f"✓ Saved BASE vs FT averages to: {out_csv}")
    return cmp_df


In [ ]:
avg_cmp_csv = "medallama_avg_comparison.csv"
cmp_df = compare_avgs("medallama_base_avg.csv",
                      "medallama_ft_avg.csv",
                      avg_cmp_csv)
display(cmp_df)


In [ ]:
import pandas as pd

# Enter your values directly
data = {
    "Metric": [
        "BERT-P", "BERT-R", "BERT-F1",
        "ROUGE-L", "FCS", "Entity F1", "MCR",
        "BLEU", "METEOR", "Hybrid",
        "LLM-Judge", "GEval"
    ],
    "Base Model": [
        0.8403, 0.8234, 0.8317,
        0.1494, 0.765, 0.3779, 0.325,
        0.0292, 0.1533, 0.5910,
        0.775, 0.675
    ],
    "Fine-Tuned Model": [
        0.8451, 0.8279, 0.8363,
        0.1706, 0.7173, 0.4816, 0.4375,
        0.0274, 0.1876, 0.5936,
        0.79, 0.80
    ]
}

# Create DataFrame
comparison_df = pd.DataFrame(data)

# Add improvement column
comparison_df["Δ (FT - Base)"] = (comparison_df["Fine-Tuned Model"] - comparison_df["Base Model"]).round(4)

# Display
print("\n=== Average Metrics: Base vs Fine-Tuned ===")
display(comparison_df)
